# Introduction

Predicting at-risk students early enables timely intervention but requires an analysis-ready table that integrates demographic, behavioural and assessment evidence without temporal or group leakage. This notebook builds that table from the seven raw OULAD CSVs and is the executable counterpart of the Chapter-3 deliverables. It calls the tested modules in `src/data` so the notebook and the committed code cannot drift.

**Target definition (Step-0, Option A).** `at_risk = 1` if `final_result ∈ {Fail, Withdrawn}`, else `0`; the label is fixed across all checkpoints.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
TABLES = ROOT / 'reports' / 'tables'
FIGS = ROOT / 'reports' / 'figures'
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
from src.data.build_master_table import build_master
from src.data.io_utils import INTERIM_DIR

# Method: integration at the student-module-presentation grain

`studentInfo` is the base table. Registration, the aggregated engagement features (from the 10.6M-row `studentVle` clickstream) and the aggregated performance features (from `studentAssessment`) are attached by **left joins**, each validated `many_to_one`. A before/after row-count log proves no duplication or loss.

In [ ]:
master = build_master(rebuild_engagement=False)
print('master_raw:', master.shape)

# Join-integrity evidence

Every step preserves exactly 32,593 records — the aggregation collapsed the one-to-many event tables to the student grain *before* joining, so no row was duplicated.

In [ ]:
display(pd.read_csv(INTERIM_DIR / 'master_join_log.csv'))

# Cleaning: deduplication, consistency, missing values, outliers

Duplicate composite keys are dropped (0 remain); categorical text is standardised; `imd_band` gaps become `Unknown`; assessment gaps become 0 plus the informative `not_submitted` flag; outliers are transformed (log1p / winsorize), never deleted.

In [ ]:
display(pd.read_csv(INTERIM_DIR / 'master_cleaning_log.csv'))

# Output validation

We confirm population size, absence of duplicate keys, and the fixed at-risk rate.

In [ ]:
print('rows:', len(master), '| columns:', master.shape[1])
print('at_risk rate: {:.1%}'.format(master['at_risk'].mean()))
key = ['code_module','code_presentation','id_student']
assert len(master) == 32593 and master.duplicated(key).sum() == 0
print('OK — 32,593 unique records, no duplicate keys')
master.head(3)

# Conclusion

The master table (32,593 × 33) integrates the three feature groups and the fixed label with audited integrity, and is the single input to the time-aware checkpoints (`src/data/make_checkpoints`) and the EDA (notebook 02).